# 1. APOE preprocessing

This notebook is part of the reproducible data-preparation pipeline used by the downstream modelling notebooks.


## 1.1. Connect Google Drive and define the APOE file path

This step mounts Google Drive in Colab and defines the exact location of the raw APOE genotype file. The source file will remain unchanged throughout preprocessing.

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount("/content/drive")

# Define the raw APOE genotype file path
apoe_file = Path(
    "/content/drive/MyDrive/adni_mri/adni_non_imaging/raw/"
    "APOE Genotype/All_Subjects_APOERES_11Jul2026.csv"
)

# Verify that the file exists
print(f"APOE file path:\n{apoe_file}")
print(f"\nFile exists: {apoe_file.exists()}")

if not apoe_file.exists():
    raise FileNotFoundError(
        "The APOE file was not found at the expected path. "
        "Check the folder and filename in Google Drive."
    )

## 1.2. Load the raw APOE genotype table

This step loads the APOE CSV into a pandas DataFrame and performs an initial structural inspection. We will check the table dimensions, column names, data types, and a small sample before interpreting any genotype coding.

In [ ]:
import pandas as pd

# Load the raw APOE genotype file
apoe_raw = pd.read_csv(apoe_file, low_memory=False)

print("APOE table loaded successfully.")
print(f"Rows: {apoe_raw.shape[0]:,}")
print(f"Columns: {apoe_raw.shape[1]:,}")

print("\nColumn names:")
print(apoe_raw.columns.tolist())

print("\nData types:")
display(apoe_raw.dtypes.to_frame(name="dtype"))

print("\nFirst 10 rows:")
display(apoe_raw.head(10))

## 1.3. Load the ADNI data dictionary and extract APOE-related definitions

This step loads the known ADNI data dictionary file and searches for entries corresponding to the columns in the APOE table. We will use these official definitions to verify the meaning and coding of `GENOTYPE` and the related sample-quality fields.

In [ ]:
from pathlib import Path
import pandas as pd

datadic_file = Path(
    "/content/drive/MyDrive/adni_mri/adni_non_imaging/raw/"
    "Cohort, dates and source-of-truth tables/DATADIC_11Jul2026.csv"
)

if not datadic_file.exists():
    raise FileNotFoundError(f"Data dictionary not found:\n{datadic_file}")

datadic = pd.read_csv(datadic_file, low_memory=False)

print("Data dictionary loaded successfully.")
print(f"Rows: {datadic.shape[0]:,}")
print(f"Columns: {datadic.shape[1]:,}")

print("\nDictionary column names:")
print(datadic.columns.tolist())

# Find dictionary rows referring to APOE table variables
apoe_columns = apoe_raw.columns.tolist()

text_view = datadic.astype("string").fillna("")
matching_mask = text_view.apply(
    lambda col: col.str.upper().isin([name.upper() for name in apoe_columns])
).any(axis=1)

apoe_dictionary_rows = datadic.loc[matching_mask].copy()

print(f"\nDictionary rows matching APOE columns: {len(apoe_dictionary_rows):,}")
display(apoe_dictionary_rows)

## 1.4. Identify the APOE genotype table in the data dictionary

The earlier dictionary search returned many unrelated tables because identifiers such as `RID`, `PTID`, and `VISCODE` occur throughout ADNI.

This step searches the dictionary table names and form names for APOE-related entries. Once the correct source table is identified, its variable definitions and coding can be inspected without mixing them with unrelated datasets.

In [ ]:
# Search for APOE-related table and form names in the ADNI data dictionary
apoe_table_candidates = (
    datadic.loc[
        datadic["TBLNAME"].astype("string").str.contains(
            "APOE|APOERES", case=False, na=False
        )
        |
        datadic["CRFNAME"].astype("string").str.contains(
            "APOE|APOERES", case=False, na=False
        ),
        ["PHASE", "CRFNAME", "TBLNAME"]
    ]
    .drop_duplicates()
    .sort_values(["TBLNAME", "PHASE"], na_position="last")
    .reset_index(drop=True)
)

print(f"Unique APOE-related dictionary table/form entries: "
      f"{len(apoe_table_candidates):,}")

display(apoe_table_candidates)

## 1.5. Inspect the official APOERES variable definitions

The correct data-dictionary table is `APOERES`, matching the raw file name. This step extracts all official variable definitions for `APOERES` across ADNI phases so that genotype and quality-control fields can be interpreted from the documented coding.

In [ ]:
apoe_dictionary = (
    datadic.loc[
        datadic["TBLNAME"].astype("string").str.upper().eq("APOERES")
    ]
    .copy()
    .sort_values(["FLDNAME", "PHASE"], na_position="last")
    .reset_index(drop=True)
)

print(f"APOERES dictionary rows: {len(apoe_dictionary):,}")
print(
    f"Variables documented: "
    f"{apoe_dictionary['FLDNAME'].nunique(dropna=True):,}"
)

display(
    apoe_dictionary[
        [
            "PHASE",
            "FLDNAME",
            "TEXT",
            "TYPE",
            "LENGTH",
            "CODE",
            "UNITS",
            "STATUS",
            "CODE_CHANGES",
            "MAPPING_NOTES",
        ]
    ]
)

## 1.6. Audit raw identifiers and genotype values

The data dictionary confirms that `GENOTYPE` is derived from two APOE alleles, each expected to be coded as 2, 3, or 4, and formatted in ascending order.

This step checks:

- missing or duplicated `RID` values;
- consistency between `RID` and `PTID`;
- the observed genotype values;
- genotype values that do not follow the documented `2/2` to `4/4` format;
- the number of records and unique participants in each ADNI phase.

No records are changed yet.

In [ ]:
# Documented valid APOE genotypes:
# alleles must be 2, 3, or 4, and GENOTYPE should be sorted low to high
valid_genotypes = {"2/2", "2/3", "2/4", "3/3", "3/4", "4/4"}

# Standardised text view for inspection only
genotype_clean_view = (
    apoe_raw["GENOTYPE"]
    .astype("string")
    .str.strip()
)

print("RAW TABLE SUMMARY")
print("-" * 60)
print(f"Rows: {len(apoe_raw):,}")
print(f"Unique non-missing RIDs: {apoe_raw['RID'].nunique(dropna=True):,}")
print(f"Missing RID values: {apoe_raw['RID'].isna().sum():,}")
print(f"Rows belonging to duplicated RIDs: "
      f"{apoe_raw.duplicated('RID', keep=False).sum():,}")
print(f"RIDs appearing more than once: "
      f"{(apoe_raw.groupby('RID').size() > 1).sum():,}")

print("\nGENOTYPE VALUE DISTRIBUTION")
print("-" * 60)
display(
    genotype_clean_view
    .value_counts(dropna=False)
    .rename_axis("GENOTYPE")
    .reset_index(name="ROW_COUNT")
)

invalid_genotype_mask = (
    genotype_clean_view.notna()
    & ~genotype_clean_view.isin(valid_genotypes)
)

print("\nGENOTYPE VALIDITY")
print("-" * 60)
print(f"Missing genotype rows: {genotype_clean_view.isna().sum():,}")
print(f"Valid documented genotype rows: "
      f"{genotype_clean_view.isin(valid_genotypes).sum():,}")
print(f"Invalid or unexpected genotype rows: "
      f"{invalid_genotype_mask.sum():,}")

if invalid_genotype_mask.any():
    print("\nUnexpected genotype records:")
    display(
        apoe_raw.loc[
            invalid_genotype_mask,
            ["PHASE", "PTID", "RID", "VISCODE", "GENOTYPE", "APTESTDT"]
        ].sort_values(["RID", "APTESTDT"])
    )

print("\nRID–PTID CONSISTENCY")
print("-" * 60)

rid_to_ptid_counts = (
    apoe_raw.dropna(subset=["RID", "PTID"])
    .groupby("RID")["PTID"]
    .nunique()
)

ptid_to_rid_counts = (
    apoe_raw.dropna(subset=["RID", "PTID"])
    .groupby("PTID")["RID"]
    .nunique()
)

print(f"RIDs linked to more than one PTID: "
      f"{(rid_to_ptid_counts > 1).sum():,}")
print(f"PTIDs linked to more than one RID: "
      f"{(ptid_to_rid_counts > 1).sum():,}")

print("\nPHASE COVERAGE")
print("-" * 60)
phase_summary = (
    apoe_raw.groupby("PHASE", dropna=False)
    .agg(
        ROWS=("RID", "size"),
        UNIQUE_RIDS=("RID", "nunique"),
        MISSING_GENOTYPE=("GENOTYPE", lambda x: x.isna().sum()),
    )
    .reset_index()
)

display(phase_summary)

## 1.7. Audit APOE sample-quality and laboratory fields

Although the genotype values are complete and valid, the raw table also contains laboratory and sample-handling variables. These fields may identify unusable samples, requested resampling, delayed receipt, or ambient-temperature shipment.

This step inspects the observed values and missingness of:

- `APRECEIVE`: received within 24 hours;
- `APAMBTEMP`: shipped at ambient temperature;
- `APRESAMP`: resample requested;
- `APUSABLE`: sample usable;
- `APVOLUME`: blood volume shipped;
- `APTESTDT`: test date.

No participant is excluded at this stage. The purpose is to determine whether QC flags should be retained in the participant-level output.

In [ ]:
qc_columns = [
    "APRECEIVE",
    "APAMBTEMP",
    "APRESAMP",
    "APUSABLE",
    "APVOLUME",
    "APTESTDT",
]

print("QC FIELD MISSINGNESS")
print("-" * 60)

qc_missingness = pd.DataFrame(
    {
        "COLUMN": qc_columns,
        "NON_MISSING": [apoe_raw[col].notna().sum() for col in qc_columns],
        "MISSING": [apoe_raw[col].isna().sum() for col in qc_columns],
        "MISSING_PERCENT": [
            apoe_raw[col].isna().mean() * 100 for col in qc_columns
        ],
    }
)

display(qc_missingness)

print("\nBINARY QC VALUE DISTRIBUTIONS")
print("-" * 60)

for col in ["APRECEIVE", "APAMBTEMP", "APRESAMP", "APUSABLE"]:
    print(f"\n{col}")
    display(
        apoe_raw[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="ROW_COUNT")
    )

print("\nQC AVAILABILITY BY PHASE")
print("-" * 60)

phase_qc_summary = (
    apoe_raw.groupby("PHASE", dropna=False)
    .agg(
        ROWS=("RID", "size"),
        APRECEIVE_AVAILABLE=("APRECEIVE", lambda x: x.notna().sum()),
        APAMBTEMP_AVAILABLE=("APAMBTEMP", lambda x: x.notna().sum()),
        APRESAMP_AVAILABLE=("APRESAMP", lambda x: x.notna().sum()),
        APUSABLE_AVAILABLE=("APUSABLE", lambda x: x.notna().sum()),
        APVOLUME_AVAILABLE=("APVOLUME", lambda x: x.notna().sum()),
        APTESTDT_AVAILABLE=("APTESTDT", lambda x: x.notna().sum()),
    )
    .reset_index()
)

display(phase_qc_summary)

print("\nPOTENTIAL SAMPLE-QUALITY FLAGS")
print("-" * 60)

potential_qc_flags = pd.DataFrame(
    {
        "FLAG": [
            "APRECEIVE != 1",
            "APAMBTEMP == 1",
            "APRESAMP == 1",
            "APUSABLE == 0",
        ],
        "ROW_COUNT": [
            ((apoe_raw["APRECEIVE"].notna()) & (apoe_raw["APRECEIVE"] != 1)).sum(),
            (apoe_raw["APAMBTEMP"] == 1).sum(),
            (apoe_raw["APRESAMP"] == 1).sum(),
            (apoe_raw["APUSABLE"] == 0).sum(),
        ],
    }
)

display(potential_qc_flags)

flagged_mask = (
    (
        apoe_raw["APRECEIVE"].notna()
        & (apoe_raw["APRECEIVE"] != 1)
    )
    | (apoe_raw["APAMBTEMP"] == 1)
    | (apoe_raw["APRESAMP"] == 1)
    | (apoe_raw["APUSABLE"] == 0)
)

print(f"\nRows with at least one potential sample-quality flag: "
      f"{flagged_mask.sum():,}")

if flagged_mask.any():
    display(
        apoe_raw.loc[
            flagged_mask,
            [
                "PHASE",
                "PTID",
                "RID",
                "GENOTYPE",
                "APTESTDT",
                "APRECEIVE",
                "APAMBTEMP",
                "APRESAMP",
                "APUSABLE",
                "APVOLUME",
            ],
        ].sort_values(["PHASE", "RID"])
    )

## 1.8. Review meaningful sample-quality combinations

The earlier provisional QC flag treated ambient-temperature shipment as an adverse condition. However, the ADNI data dictionary defines `APAMBTEMP` only as a descriptive yes/no variable and does not identify `1` as a QC failure.

For genotype usability, the most important documented fields are:

- `APUSABLE = 0`: the sample was marked unusable;
- `APRESAMP = 1`: a new sample was requested.

`APRECEIVE = 0` and unusual shipping conditions will be retained as descriptive QC information but will not invalidate a genotype when the sample is marked usable and no resampling was requested.

This step examines the exact combinations of these fields and isolates the small number of unusual ADNI1 records for review.

In [ ]:
# Summarise observed ADNI1 laboratory QC combinations
adni1_qc_combinations = (
    apoe_raw.loc[
        apoe_raw["PHASE"].eq("ADNI1"),
        ["APRECEIVE", "APAMBTEMP", "APRESAMP", "APUSABLE"],
    ]
    .value_counts(dropna=False)
    .rename("ROW_COUNT")
    .reset_index()
    .sort_values("ROW_COUNT", ascending=False)
    .reset_index(drop=True)
)

print("ADNI1 QC COMBINATIONS")
print("-" * 60)
display(adni1_qc_combinations)

# Meaningful adverse indicators based on the documented field meanings
unusable_mask = apoe_raw["APUSABLE"].eq(0)
resample_mask = apoe_raw["APRESAMP"].eq(1)

# Descriptive irregularities that do not automatically invalidate genotype
late_receipt_mask = apoe_raw["APRECEIVE"].eq(0)
non_ambient_mask = apoe_raw["APAMBTEMP"].eq(0)

print("\nQC REVIEW COUNTS")
print("-" * 60)
print(f"Samples marked unusable: {unusable_mask.sum():,}")
print(f"Samples requiring resampling: {resample_mask.sum():,}")
print(f"Samples not received within 24 hours: {late_receipt_mask.sum():,}")
print(f"Samples not shipped at ambient temperature: {non_ambient_mask.sum():,}")

# Inspect records with either descriptive irregularity
unusual_shipping_mask = late_receipt_mask | non_ambient_mask

print(
    f"\nRows with late receipt and/or non-ambient shipment: "
    f"{unusual_shipping_mask.sum():,}"
)

display(
    apoe_raw.loc[
        unusual_shipping_mask,
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "GENOTYPE",
            "APTESTDT",
            "APRECEIVE",
            "APAMBTEMP",
            "APRESAMP",
            "APUSABLE",
            "APVOLUME",
        ],
    ]
    .sort_values(["APRECEIVE", "APAMBTEMP", "RID"])
    .reset_index(drop=True)
)

# Check whether any unusual record was also unusable or required resampling
unusual_with_failure_mask = unusual_shipping_mask & (
    unusable_mask | resample_mask
)

print(
    "\nUnusual shipping/receipt records also marked unusable "
    f"or requiring resampling: {unusual_with_failure_mask.sum():,}"
)

## 1.9. Construct the cleaned participant-level APOE table

The raw APOE table already contains exactly one record per `RID`, with no duplicate participants, missing genotypes, invalid allele combinations, or repeated-record conflicts.

This step creates the modelling-ready participant-level table by:

- standardising identifiers and text fields;
- splitting `GENOTYPE` into `APOE_ALLELE_1` and `APOE_ALLELE_2`;
- retaining the canonical low-to-high genotype representation;
- calculating the number of APOE ε4 alleles;
- defining `APOE4_CARRIER` as at least one ε4 allele;
- preserving available ADNI1 sample-quality information;
- adding explicit QC flags without excluding valid genotype results.

Because APOE genotype is stable, no visit-date or baseline-window filtering is applied.

In [ ]:
# Create the cleaned participant-level APOE table
apoe_cleaned = apoe_raw.copy()

# Standardise identifier and descriptive fields
apoe_cleaned["PHASE"] = (
    apoe_cleaned["PHASE"]
    .astype("string")
    .str.strip()
)

apoe_cleaned["PTID"] = (
    apoe_cleaned["PTID"]
    .astype("string")
    .str.strip()
)

apoe_cleaned["RID"] = pd.to_numeric(
    apoe_cleaned["RID"], errors="coerce"
).astype("Int64")

apoe_cleaned["VISCODE"] = (
    apoe_cleaned["VISCODE"]
    .astype("string")
    .str.strip()
)

apoe_cleaned["GENOTYPE"] = (
    apoe_cleaned["GENOTYPE"]
    .astype("string")
    .str.strip()
)

apoe_cleaned["APTESTDT"] = pd.to_datetime(
    apoe_cleaned["APTESTDT"],
    errors="coerce"
)

# Split the documented low-to-high genotype representation
alleles = apoe_cleaned["GENOTYPE"].str.split("/", expand=True)

apoe_cleaned["APOE_ALLELE_1"] = pd.to_numeric(
    alleles[0], errors="coerce"
).astype("Int64")

apoe_cleaned["APOE_ALLELE_2"] = pd.to_numeric(
    alleles[1], errors="coerce"
).astype("Int64")

# Reconstruct genotype from the parsed alleles
apoe_cleaned["APOE_GENOTYPE"] = (
    apoe_cleaned["APOE_ALLELE_1"].astype("string")
    + "/"
    + apoe_cleaned["APOE_ALLELE_2"].astype("string")
)

# Derive APOE ε4 features
apoe_cleaned["APOE4_ALLELE_COUNT"] = (
    apoe_cleaned[["APOE_ALLELE_1", "APOE_ALLELE_2"]]
    .eq(4)
    .sum(axis=1)
    .astype("Int64")
)

apoe_cleaned["APOE4_CARRIER"] = (
    apoe_cleaned["APOE4_ALLELE_COUNT"] >= 1
).astype("boolean")

# QC flags
apoe_cleaned["APOE_GENOTYPE_VALID"] = (
    apoe_cleaned["APOE_GENOTYPE"].isin(valid_genotypes)
).astype("boolean")

apoe_cleaned["APOE_SAMPLE_USABLE"] = (
    apoe_cleaned["APUSABLE"]
    .map({1.0: True, 0.0: False})
    .astype("boolean")
)

apoe_cleaned["APOE_RESAMPLE_REQUESTED"] = (
    apoe_cleaned["APRESAMP"]
    .map({1.0: True, 0.0: False})
    .astype("boolean")
)

apoe_cleaned["APOE_RECEIVED_WITHIN_24H"] = (
    apoe_cleaned["APRECEIVE"]
    .map({1.0: True, 0.0: False})
    .astype("boolean")
)

apoe_cleaned["APOE_SHIPPED_AMBIENT"] = (
    apoe_cleaned["APAMBTEMP"]
    .map({1.0: True, 0.0: False})
    .astype("boolean")
)

# Structural absence of laboratory QC fields outside ADNI1
apoe_cleaned["APOE_LAB_QC_AVAILABLE"] = (
    apoe_cleaned[
        ["APRECEIVE", "APAMBTEMP", "APRESAMP", "APUSABLE", "APTESTDT"]
    ]
    .notna()
    .any(axis=1)
    .astype("boolean")
)

# No conflicts were observed because each RID occurs exactly once
apoe_cleaned["APOE_UNRESOLVED_CONFLICT"] = False
apoe_cleaned["APOE_UNRESOLVED_CONFLICT"] = (
    apoe_cleaned["APOE_UNRESOLVED_CONFLICT"].astype("boolean")
)

# Select modelling and QC columns only
apoe_cleaned = apoe_cleaned[
    [
        "PHASE",
        "PTID",
        "RID",
        "VISCODE",
        "APTESTDT",
        "APOE_ALLELE_1",
        "APOE_ALLELE_2",
        "APOE_GENOTYPE",
        "APOE4_ALLELE_COUNT",
        "APOE4_CARRIER",
        "APOE_GENOTYPE_VALID",
        "APOE_LAB_QC_AVAILABLE",
        "APOE_SAMPLE_USABLE",
        "APOE_RESAMPLE_REQUESTED",
        "APOE_RECEIVED_WITHIN_24H",
        "APOE_SHIPPED_AMBIENT",
        "APOE_UNRESOLVED_CONFLICT",
    ]
].sort_values("RID").reset_index(drop=True)

print("Cleaned APOE participant-level table created.")
print(f"Rows: {len(apoe_cleaned):,}")
print(f"Unique RIDs: {apoe_cleaned['RID'].nunique(dropna=True):,}")
print(f"Duplicate RIDs: {apoe_cleaned.duplicated('RID').sum():,}")

print("\nData types:")
display(apoe_cleaned.dtypes.to_frame(name="DTYPE"))

print("\nFirst 10 rows:")
display(apoe_cleaned.head(10))

## 1.10. Validate the derived APOE variables

Before saving the participant-level table, this step verifies that:

- every parsed allele is one of 2, 3, or 4;
- the two alleles remain sorted from low to high;
- the reconstructed genotype matches the original genotype;
- ε4 allele counts agree with the genotype;
- carrier status agrees with an ε4 allele count of at least one;
- no duplicate or missing `RID` values were introduced;
- no unresolved conflicts or invalid genotypes remain.

These checks ensure that the derived modelling variables are internally consistent.

In [ ]:
# Validate participant identifiers
duplicate_rids = apoe_cleaned.duplicated("RID", keep=False)
missing_rids = apoe_cleaned["RID"].isna()

# Validate allele coding and ordering
valid_allele_values = {2, 3, 4}

invalid_allele_mask = (
    ~apoe_cleaned["APOE_ALLELE_1"].isin(valid_allele_values)
    | ~apoe_cleaned["APOE_ALLELE_2"].isin(valid_allele_values)
)

unsorted_allele_mask = (
    apoe_cleaned["APOE_ALLELE_1"]
    > apoe_cleaned["APOE_ALLELE_2"]
)

# Confirm reconstructed genotype matches the raw standardised genotype
genotype_mismatch_mask = (
    apoe_cleaned["APOE_GENOTYPE"]
    != apoe_raw.loc[
        apoe_cleaned.index, "GENOTYPE"
    ].astype("string").str.strip()
)

# Recalculate ε4 count independently
expected_e4_count = (
    apoe_cleaned["APOE_ALLELE_1"].eq(4).astype("Int64")
    + apoe_cleaned["APOE_ALLELE_2"].eq(4).astype("Int64")
)

e4_count_mismatch_mask = (
    apoe_cleaned["APOE4_ALLELE_COUNT"] != expected_e4_count
)

expected_carrier = expected_e4_count.ge(1)
carrier_mismatch_mask = (
    apoe_cleaned["APOE4_CARRIER"] != expected_carrier
)

validation_summary = pd.DataFrame(
    {
        "CHECK": [
            "Missing RID",
            "Rows with duplicated RID",
            "Invalid allele values",
            "Alleles not sorted low to high",
            "Reconstructed genotype mismatch",
            "Invalid genotype flag",
            "ε4 allele-count mismatch",
            "APOE4 carrier-status mismatch",
            "Unresolved genotype conflict",
        ],
        "ISSUE_COUNT": [
            missing_rids.sum(),
            duplicate_rids.sum(),
            invalid_allele_mask.sum(),
            unsorted_allele_mask.sum(),
            genotype_mismatch_mask.sum(),
            (~apoe_cleaned["APOE_GENOTYPE_VALID"]).sum(),
            e4_count_mismatch_mask.sum(),
            carrier_mismatch_mask.sum(),
            apoe_cleaned["APOE_UNRESOLVED_CONFLICT"].sum(),
        ],
    }
)

print("APOE DERIVED-VARIABLE VALIDATION")
print("-" * 60)
display(validation_summary)

total_issues = validation_summary["ISSUE_COUNT"].sum()

print(f"\nTotal validation issues: {total_issues:,}")

if total_issues == 0:
    print("All participant-level APOE validation checks passed.")
else:
    print("One or more validation checks failed. Review before saving.")

print("\nGENOTYPE DISTRIBUTION")
print("-" * 60)
display(
    apoe_cleaned["APOE_GENOTYPE"]
    .value_counts(dropna=False)
    .rename_axis("APOE_GENOTYPE")
    .reset_index(name="PARTICIPANTS")
)

print("\nAPOE ε4 ALLELE-COUNT DISTRIBUTION")
print("-" * 60)
display(
    apoe_cleaned["APOE4_ALLELE_COUNT"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("APOE4_ALLELE_COUNT")
    .reset_index(name="PARTICIPANTS")
)

print("\nAPOE ε4 CARRIER DISTRIBUTION")
print("-" * 60)
display(
    apoe_cleaned["APOE4_CARRIER"]
    .value_counts(dropna=False)
    .rename_axis("APOE4_CARRIER")
    .reset_index(name="PARTICIPANTS")
)

## 1.11. Correct the genotype comparison using `RID`

The previous validation compared the cleaned and raw genotype columns by row position. This was inappropriate because the cleaned table had been sorted by `RID` and its index reset.

This step performs the comparison correctly by joining the cleaned and raw genotype values using the participant identifier `RID`. It then repeats the complete validation summary.

No genotype values are changed.

In [ ]:
# Prepare the original genotype values using RID as the alignment key
raw_genotype_by_rid = (
    apoe_raw[["RID", "GENOTYPE"]]
    .copy()
)

raw_genotype_by_rid["RID"] = pd.to_numeric(
    raw_genotype_by_rid["RID"],
    errors="coerce"
).astype("Int64")

raw_genotype_by_rid["RAW_GENOTYPE"] = (
    raw_genotype_by_rid["GENOTYPE"]
    .astype("string")
    .str.strip()
)

raw_genotype_by_rid = raw_genotype_by_rid[
    ["RID", "RAW_GENOTYPE"]
]

# Align raw and cleaned genotype values by RID
genotype_validation = apoe_cleaned[
    ["RID", "APOE_GENOTYPE"]
].merge(
    raw_genotype_by_rid,
    on="RID",
    how="left",
    validate="one_to_one",
)

genotype_validation["GENOTYPE_MATCH"] = (
    genotype_validation["APOE_GENOTYPE"]
    == genotype_validation["RAW_GENOTYPE"]
)

genotype_mismatch_mask = ~genotype_validation["GENOTYPE_MATCH"]

# Repeat the other validation checks
duplicate_rids = apoe_cleaned.duplicated("RID", keep=False)
missing_rids = apoe_cleaned["RID"].isna()

valid_allele_values = {2, 3, 4}

invalid_allele_mask = (
    ~apoe_cleaned["APOE_ALLELE_1"].isin(valid_allele_values)
    | ~apoe_cleaned["APOE_ALLELE_2"].isin(valid_allele_values)
)

unsorted_allele_mask = (
    apoe_cleaned["APOE_ALLELE_1"]
    > apoe_cleaned["APOE_ALLELE_2"]
)

expected_e4_count = (
    apoe_cleaned["APOE_ALLELE_1"].eq(4).astype("Int64")
    + apoe_cleaned["APOE_ALLELE_2"].eq(4).astype("Int64")
)

e4_count_mismatch_mask = (
    apoe_cleaned["APOE4_ALLELE_COUNT"]
    != expected_e4_count
)

expected_carrier = expected_e4_count.ge(1)

carrier_mismatch_mask = (
    apoe_cleaned["APOE4_CARRIER"]
    != expected_carrier
)

corrected_validation_summary = pd.DataFrame(
    {
        "CHECK": [
            "Missing RID",
            "Rows with duplicated RID",
            "Invalid allele values",
            "Alleles not sorted low to high",
            "Reconstructed genotype mismatch after RID alignment",
            "Invalid genotype flag",
            "ε4 allele-count mismatch",
            "APOE4 carrier-status mismatch",
            "Unresolved genotype conflict",
        ],
        "ISSUE_COUNT": [
            missing_rids.sum(),
            duplicate_rids.sum(),
            invalid_allele_mask.sum(),
            unsorted_allele_mask.sum(),
            genotype_mismatch_mask.sum(),
            (~apoe_cleaned["APOE_GENOTYPE_VALID"]).sum(),
            e4_count_mismatch_mask.sum(),
            carrier_mismatch_mask.sum(),
            apoe_cleaned["APOE_UNRESOLVED_CONFLICT"].sum(),
        ],
    }
)

print("CORRECTED APOE VALIDATION")
print("-" * 60)
display(corrected_validation_summary)

total_corrected_issues = corrected_validation_summary[
    "ISSUE_COUNT"
].sum()

print(f"\nTotal validation issues: {total_corrected_issues:,}")

if total_corrected_issues == 0:
    print("All corrected APOE validation checks passed.")
else:
    print("One or more genuine validation issues remain.")

if genotype_mismatch_mask.any():
    print("\nGenotype mismatches after RID alignment:")
    display(
        genotype_validation.loc[
            genotype_mismatch_mask
        ].sort_values("RID")
    )

## 1.12. Save the cleaned APOE table and produce the final QC report

This step saves the validated participant-level APOE dataset to the processed APOE directory.

The final report summarises:

- total rows and unique participants;
- duplicate and missing `RID` values;
- missing or invalid genotype values;
- genotype distribution;
- APOE ε4 allele-count distribution;
- APOE ε4 carrier distribution;
- unresolved conflicts;
- available laboratory QC information.

The output contains no diagnosis, outcome, or cohort-label variables.

In [ ]:
from pathlib import Path

# Define and create the output directory
apoe_output_dir = Path(
    "/content/drive/MyDrive/adni_mri/adni_non_imaging/"
    "processed/apoe"
)
apoe_output_dir.mkdir(parents=True, exist_ok=True)

# Define the final output file
apoe_output_file = (
    apoe_output_dir
    / "apoe_genotype_cleaned_participant_level.csv"
)

# Save the cleaned participant-level table
apoe_cleaned.to_csv(
    apoe_output_file,
    index=False,
    date_format="%Y-%m-%d"
)

print("FINAL APOE DATASET SAVED")
print("-" * 70)
print(f"Output file:\n{apoe_output_file}")
print(f"File exists: {apoe_output_file.exists()}")
print(f"File size: {apoe_output_file.stat().st_size / 1024:.2f} KB")

# Reload the saved file to verify successful export
apoe_saved_check = pd.read_csv(
    apoe_output_file,
    low_memory=False
)

print("\nSAVED FILE VERIFICATION")
print("-" * 70)
print(f"Saved rows: {len(apoe_saved_check):,}")
print(f"Saved columns: {apoe_saved_check.shape[1]:,}")
print(f"Saved unique RIDs: {apoe_saved_check['RID'].nunique(dropna=True):,}")
print(f"Saved duplicate RIDs: {apoe_saved_check.duplicated('RID').sum():,}")

# Final headline QC report
final_qc_report = pd.DataFrame(
    {
        "METRIC": [
            "Rows",
            "Unique participants",
            "Missing RID",
            "Duplicate RID",
            "Missing genotype",
            "Invalid genotype",
            "Missing allele 1",
            "Missing allele 2",
            "Unresolved conflicts",
            "Laboratory QC available",
            "Samples marked unusable",
            "Resampling requested",
            "Not received within 24 hours",
            "Not shipped at ambient temperature",
        ],
        "COUNT": [
            len(apoe_cleaned),
            apoe_cleaned["RID"].nunique(dropna=True),
            apoe_cleaned["RID"].isna().sum(),
            apoe_cleaned.duplicated("RID").sum(),
            apoe_cleaned["APOE_GENOTYPE"].isna().sum(),
            (~apoe_cleaned["APOE_GENOTYPE_VALID"]).sum(),
            apoe_cleaned["APOE_ALLELE_1"].isna().sum(),
            apoe_cleaned["APOE_ALLELE_2"].isna().sum(),
            apoe_cleaned["APOE_UNRESOLVED_CONFLICT"].sum(),
            apoe_cleaned["APOE_LAB_QC_AVAILABLE"].sum(),
            apoe_cleaned["APOE_SAMPLE_USABLE"].eq(False).sum(),
            apoe_cleaned["APOE_RESAMPLE_REQUESTED"].eq(True).sum(),
            apoe_cleaned["APOE_RECEIVED_WITHIN_24H"].eq(False).sum(),
            apoe_cleaned["APOE_SHIPPED_AMBIENT"].eq(False).sum(),
        ],
    }
)

print("\nFINAL QC REPORT")
print("-" * 70)
display(final_qc_report)

print("\nGENOTYPE DISTRIBUTION")
print("-" * 70)
display(
    apoe_cleaned["APOE_GENOTYPE"]
    .value_counts(dropna=False)
    .rename_axis("APOE_GENOTYPE")
    .reset_index(name="PARTICIPANTS")
)

print("\nAPOE ε4 ALLELE-COUNT DISTRIBUTION")
print("-" * 70)
display(
    apoe_cleaned["APOE4_ALLELE_COUNT"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("APOE4_ALLELE_COUNT")
    .reset_index(name="PARTICIPANTS")
)

print("\nAPOE ε4 CARRIER DISTRIBUTION")
print("-" * 70)
display(
    apoe_cleaned["APOE4_CARRIER"]
    .value_counts(dropna=False)
    .rename_axis("APOE4_CARRIER")
    .reset_index(name="PARTICIPANTS")
)

print("\nAPOE preprocessing completed successfully.")